<a href="https://colab.research.google.com/github/yassinemaataoui/Colab_project/blob/main/EcoWattSpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **la configuration**

In [ ]:
!apt-get update -qq && apt-get install -y openjdk-11-jdk
version = "3.5.7"
hadoop_version = "hadoop3"
url = f"https://downloads.apache.org/spark/spark-{version}/spark-{version}-bin-{hadoop_version}.tgz"
!wget {url} -O spark-{version}-bin-{hadoop_version}.tgz
!ls -lh spark-{version}-bin-{hadoop_version}.tgz
!tar -xvzf spark-{version}-bin-{hadoop_version}.tgz
!pip install -q findspark
import os, findspark
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = f"/content/spark-{version}-bin-{hadoop_version}"
findspark.init()

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-11-jdk is already the newest version (11.0.28+6-1ubuntu1~22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.
--2025-11-13 14:28:46--  https://downloads.apache.org/spark/spark-3.5.7/spark-3.5.7-bin-hadoop3.tgz
Resolving downloads.apache.org (downloads.apache.org)... 88.99.208.237, 135.181.214.104, 2a01:4f9:3a:2c57::2, ...
Connecting to downloads.apache.org (downloads.apache.org)|88.99.208.237|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 400914067 (382M) [application/x-gzip]
Saving to: ‘spark-3.5.7-bin-hadoop3.tgz’

spark-3.5.7-bin-had 100%[===================>] 382.34M  16.6MB/s    in 26s     

2025-11-13 14:29:13 (14.6 MB/s) - ‘spark-3.5.7-bin-had

### **Créer la session Spark (nom : EcoWattSpark)**

In [ ]:
spark = SparkSession.builder \
    .appName("EcoWattSpark") \
    .getOrCreate()

Dans cette étape, le script commence par importer les bibliothèques PySpark nécessaires, puis crée une session Spark nommée EcoWattSpark. Cette session représente l’environnement principal dans lequel toutes les opérations de traitement et d’analyse de données vont être exécutées. Sans cette session, aucune manipulation de DataFrame n’est possible.

# **Exercice 1 : Chargement et découverte**

### **1. Lire le CSV (header=True, inferSchema=True)**

In [ ]:
input_path = "consommation_ecowatt.csv"  # adapter le chemin si besoin
df = spark.read.csv(input_path, header=True, inferSchema=True)

Le script lit le fichier consommation_ecowatt.csv grâce à la fonction spark.read.csv. Les paramètres utilisés permettent à Spark d’identifier automatiquement le type des colonnes et de reconnaître la première ligne comme un en-tête. Ensuite, le schéma des données est affiché pour vérifier la structure du fichier, et cinq premières lignes sont montrées pour avoir un aperçu du contenu.

### **Afficher le schéma**

In [ ]:
# Afficher le schéma
print("=== Schéma du DataFrame ===")
df.printSchema()

=== Schéma du DataFrame ===
root
 |-- id_mesure: integer (nullable = true)
 |-- date_heure: timestamp (nullable = true)
 |-- ville: string (nullable = true)
 |-- type_batiment: string (nullable = true)
 |-- conso_kwh: double (nullable = true)
 |-- temperature_ext: double (nullable = true)
 |-- cout_kwh: double (nullable = true)
 |-- nb_personnes: integer (nullable = true)



### **Nombre total d'enregistrements**

In [ ]:
nb_total = df.count()
print(f"Nombre total d'enregistrements : {nb_total}")

Nombre total d'enregistrements : 300


### **Afficher 5 premières lignes**

In [ ]:
print("=== 5 premières lignes ===")
df.show(5, truncate=False)

=== 5 premières lignes ===
+---------+-------------------+----------+-------------+---------+---------------+--------+------------+
|id_mesure|date_heure         |ville     |type_batiment|conso_kwh|temperature_ext|cout_kwh|nb_personnes|
+---------+-------------------+----------+-------------+---------+---------------+--------+------------+
|1        |2025-02-05 20:00:00|Marrakech |Résidentiel  |7973.1   |35.3           |1.37    |115         |
|2        |2025-02-23 22:00:00|Agadir    |École        |2096.1   |19.6           |1.3     |199         |
|3        |2025-02-17 02:00:00|Tanger    |École        |1720.2   |34.9           |1.2     |167         |
|4        |2025-02-15 15:00:00|Casablanca|Hôpital      |1726.2   |11.1           |1.23    |130         |
|5        |2025-03-10 06:00:00|Tanger    |Résidentiel  |8164.3   |27.9           |1.33    |64          |
+---------+-------------------+----------+-------------+---------+---------------+--------+------------+
only showing top 5 rows



# **Exercice 2 : Nettoyage et enrichissement**
À ce stade, le programme supprime toutes les lignes contenant au moins une valeur manquante. Cette opération garantit que les analyses futures ne seront pas faussées par des données incomplètes. Le DataFrame obtenu est ainsi plus fiable pour les étapes d’analyse suivantes.
### **1) Supprimer les valeurs nulles (sur toutes les colonnes)**

In [ ]:
df_nonull = df.na.drop()

### **2) Éliminer les mesures aberrantes :**
 conso_kwh < 0 ou conso_kwh > 20000

 temperature_ext < -10 ou temperature_ext > 50

In [ ]:
df_clean = df_nonull.filter(
    (col("conso_kwh") >= 0) &
    (col("conso_kwh") <= 20000) &
    (col("temperature_ext") >= -10) &
    (col("temperature_ext") <= 50)
)

Après avoir retiré les valeurs nulles, le script élimine les enregistrements contenant des valeurs incohérentes, comme une consommation négative ou très élevée (plus de 20 000 kWh), ou une température extérieure inférieure à –10°C ou supérieure à 50°C. Ce filtrage permet d’écarter les erreurs de mesure et d’obtenir un jeu de données propre.

### **3) Créer colonnes calculées**
 - cout_total = conso_kwh * cout_kwh
 - conso_par_personne = conso_kwh / nb_personnes (protéger   division par 0)

In [ ]:
df_enriched = df_clean.withColumn(
    "cout_total", (col("conso_kwh") * col("cout_kwh")).cast(DoubleType())
).withColumn(
    "conso_par_personne",
    when(col("nb_personnes") > 0, col("conso_kwh") / col("nb_personnes")).otherwise(None)
)

Le script enrichit ensuite les données en ajoutant deux nouvelles colonnes. La première est cout_total, obtenue en multipliant la consommation par le coût du kWh. La seconde est conso_par_personne, qui divise la consommation par le nombre d’occupants, tout en évitant la division par zéro. Cela permet d’obtenir des informations plus détaillées et utiles pour l’analyse.

### **Afficher le nombre de lignes avant/après nettoyage**

In [ ]:
print("Lignes avant nettoyage (après drop null) :", df_nonull.count())
print("Lignes après filtres aberrants :", df_enriched.count())

Lignes avant nettoyage (après drop null) : 300
Lignes après filtres aberrants : 300


### **Montrer un échantillon enrichi**

In [ ]:
print("=== Exemple lignes enrichies ===")
df_enriched.select("id_mesure", "date_heure", "ville", "type_batiment",
                   "conso_kwh", "nb_personnes", "conso_par_personne", "cout_total") \
           .show(10, truncate=False)

=== Exemple lignes enrichies ===
+---------+-------------------+----------+-------------+---------+------------+------------------+-----------------+
|id_mesure|date_heure         |ville     |type_batiment|conso_kwh|nb_personnes|conso_par_personne|cout_total       |
+---------+-------------------+----------+-------------+---------+------------+------------------+-----------------+
|1        |2025-02-05 20:00:00|Marrakech |Résidentiel  |7973.1   |115         |69.33130434782609 |10923.147        |
|2        |2025-02-23 22:00:00|Agadir    |École        |2096.1   |199         |10.533165829145728|2724.93          |
|3        |2025-02-17 02:00:00|Tanger    |École        |1720.2   |167         |10.30059880239521 |2064.24          |
|4        |2025-02-15 15:00:00|Casablanca|Hôpital      |1726.2   |130         |13.278461538461539|2123.226         |
|5        |2025-03-10 06:00:00|Tanger    |Résidentiel  |8164.3   |64          |127.5671875       |10858.519        |
|6        |2025-01-20 10:00:00|

# **Exercice 3 : Analyses descriptives**

### **1) Consommation totale par ville**

In [ ]:
conso_par_ville = df_enriched.groupBy("ville").agg(_sum("conso_kwh").alias("conso_totale_kwh"))
print("=== Consommation totale par ville ===")
conso_par_ville.orderBy(col("conso_totale_kwh").desc()).show(truncate=False)

=== Consommation totale par ville ===
+----------+------------------+
|ville     |conso_totale_kwh  |
+----------+------------------+
|Tanger    |335346.9          |
|Agadir    |257718.90000000005|
|Rabat     |244939.20000000004|
|Marrakech |232741.69999999998|
|Casablanca|217569.00000000003|
|Fès       |209143.39999999997|
+----------+------------------+



### **2) Consommation moyenne par type de bâtiment**

In [ ]:
conso_moy_par_type = df_enriched.groupBy("type_batiment").agg(_avg("conso_kwh").alias("conso_moy_kwh"))
print("=== Consommation moyenne par type de bâtiment ===")
conso_moy_par_type.orderBy(col("conso_moy_kwh").desc()).show(truncate=False)

=== Consommation moyenne par type de bâtiment ===
+-------------+-----------------+
|type_batiment|conso_moy_kwh    |
+-------------+-----------------+
|Résidentiel  |5268.075000000002|
|Bureau       |5156.969841269843|
|École        |5112.007547169812|
|Hôpital      |4711.804918032788|
|Industriel   |4695.876271186441|
+-------------+-----------------+



Le programme calcule aussi la consommation moyenne selon le type de bâtiment. Cette analyse met en évidence quels types de bâtiments sont les plus énergivores en moyenne, comme les industries, les bureaux ou les habitations.

### **3) Classement des villes de la plus énergivore à la moins énergivore**

In [ ]:
print("=== Classement villes (du plus au moins énergivore) ===")
conso_par_ville.orderBy(col("conso_totale_kwh").desc()).show(truncate=False)

=== Classement villes (du plus au moins énergivore) ===
+----------+------------------+
|ville     |conso_totale_kwh  |
+----------+------------------+
|Tanger    |335346.9          |
|Agadir    |257718.90000000005|
|Rabat     |244939.20000000004|
|Marrakech |232741.69999999998|
|Casablanca|217569.00000000003|
|Fès       |209143.39999999997|
+----------+------------------+



Les villes sont ensuite triées de la plus énergivore à la moins énergivore. Ce classement permet d’avoir une vision claire de la distribution géographique de la consommation énergétique et d’identifier les zones prioritaires.

# **Exercice 4 : Analyse temporelle**

### **1) Créer colonnes date, heure, jour (nom du jour), mois**

In [ ]:

df_time = df_enriched.withColumn("date", to_date(col("date_heure"))) \
                     .withColumn("heure", hour(col("date_heure"))) \
                     .withColumn("jour", date_format(to_date(col("date_heure")), "EEEE")) \
                     .withColumn("mois", month(col("date_heure")))

print("=== Exemple colonnes temporelles ===")
df_time.select("date_heure", "date", "heure", "jour", "mois").show(5, truncate=False)

=== Exemple colonnes temporelles ===
+-------------------+----------+-----+---------+----+
|date_heure         |date      |heure|jour     |mois|
+-------------------+----------+-----+---------+----+
|2025-02-05 20:00:00|2025-02-05|20   |Wednesday|2   |
|2025-02-23 22:00:00|2025-02-23|22   |Sunday   |2   |
|2025-02-17 02:00:00|2025-02-17|2    |Monday   |2   |
|2025-02-15 15:00:00|2025-02-15|15   |Saturday |2   |
|2025-03-10 06:00:00|2025-03-10|6    |Monday   |3   |
+-------------------+----------+-----+---------+----+
only showing top 5 rows



Le script crée de nouvelles colonnes liées au temps, comme la date simplifiée, l’heure, le jour de la semaine et le mois. Ces colonnes supplémentaires permettent d’analyser comment la consommation varie selon les périodes de la journée, les jours ou les mois.

### **2) Calculs temporels**
 - **consommation totale par mois**

In [ ]:
conso_par_mois = df_time.groupBy("mois").agg(_sum("conso_kwh").alias("conso_totale_kwh")).orderBy("mois")
print("=== Consommation totale par mois ===")
conso_par_mois.show(12, truncate=False)

=== Consommation totale par mois ===
+----+------------------+
|mois|conso_totale_kwh  |
+----+------------------+
|1   |457750.5          |
|2   |488114.30000000005|
|3   |551594.3          |
+----+------------------+



- **consommation moyenne par jour de la semaine**

In [ ]:
conso_moy_par_jour = df_time.groupBy("jour").agg(_avg("conso_kwh").alias("conso_moy_kwh")).orderBy("jour")
print("=== Consommation moyenne par jour de la semaine ===")
conso_moy_par_jour.show(7, truncate=False)

=== Consommation moyenne par jour de la semaine ===
+---------+-----------------+
|jour     |conso_moy_kwh    |
+---------+-----------------+
|Friday   |4649.504761904762|
|Monday   |5002.612500000001|
|Saturday |4463.758064516128|
|Sunday   |5476.811764705883|
|Thursday |4903.738636363636|
|Tuesday  |5118.782926829269|
|Wednesday|5227.132692307694|
+---------+-----------------+



- **consommation horaire moyenne**

In [ ]:
conso_moy_par_heure = df_time.groupBy("heure").agg(_avg("conso_kwh").alias("conso_moy_kwh")).orderBy("heure")
print("=== Consommation horaire moyenne ===")
conso_moy_par_heure.show(24, truncate=False)

=== Consommation horaire moyenne ===
+-----+------------------+
|heure|conso_moy_kwh     |
+-----+------------------+
|0    |6133.05           |
|1    |3642.4            |
|2    |4624.241666666667 |
|3    |3408.207692307692 |
|4    |4969.84           |
|5    |3913.891666666666 |
|6    |5513.11818181818  |
|7    |6501.415384615385 |
|8    |5972.026666666668 |
|9    |6239.542857142857 |
|10   |5638.747368421052 |
|11   |6063.293749999999 |
|12   |3878.4500000000007|
|13   |2629.3749999999995|
|14   |5633.25           |
|15   |6250.841666666666 |
|16   |4841.582352941176 |
|17   |4704.075          |
|18   |4623.557142857143 |
|19   |5785.14705882353  |
|20   |5222.891666666666 |
|21   |6368.285714285715 |
|22   |3462.5928571428567|
|23   |5341.242857142856 |
+-----+------------------+



À partir des colonnes temporelles, le script calcule la consommation totale par mois, puis la consommation moyenne pour chaque jour de la semaine, et enfin la consommation moyenne selon les heures de la journée. Cela permet d’identifier les périodes de forte demande énergétique.

# **Exercice 5 : Comparaison DataFrame / SQL**

### **1) Créer une vue temporaire 'conso'**

In [ ]:
df_time.createOrReplaceTempView("conso")

Le script transforme le DataFrame en vue SQL appelée conso. Cela permet d’exécuter des requêtes SQL pour obtenir les mêmes analyses que précédemment, comme la consommation totale par type de bâtiment ou la consommation moyenne par personne. Cette étape montre que Spark permet de travailler aussi bien en SQL qu’en DataFrame API.

### **2) Requêtes SQL demandées**

In [ ]:
print("=== SQL : consommation totale par type_batiment ===")
spark.sql("""
    SELECT type_batiment, SUM(conso_kwh) AS conso_totale_kwh
    FROM conso
    GROUP BY type_batiment
    ORDER BY conso_totale_kwh DESC
""").show(truncate=False)

print("=== SQL : consommation moyenne par personne ===")
spark.sql("""
    SELECT AVG(conso_kwh / NULLIF(nb_personnes,0)) AS conso_moy_par_personne
    FROM conso
""").show(truncate=False)

print("=== SQL : Top 2 bâtiments les plus énergivores par ville ===")

=== SQL : consommation totale par type_batiment ===
+-------------+-----------------+
|type_batiment|conso_totale_kwh |
+-------------+-----------------+
|Résidentiel  |337156.8000000001|
|Bureau       |324889.1000000001|
|Hôpital      |287420.1000000001|
|Industriel   |277056.7         |
|École        |270936.4         |
+-------------+-----------------+

=== SQL : consommation moyenne par personne ===
+----------------------+
|conso_moy_par_personne|
+----------------------+
|79.4067845188413      |
+----------------------+

=== SQL : Top 2 bâtiments les plus énergivores par ville ===


 Supposons qu'il y a un identifiant bâtiment (ici on n'a que type & ville ; si on a 'id_batiment' remplacez id_mesure par id_batiment)

 On fera la top 2 par ville basé sur la somme par id_mesure / bâtiment si on avait un id_batiment.

 Exemple générique en présumant 'id_batiment' existe; sinon on utilise 'id_mesure' (moins pertinent).

In [ ]:
spark.sql("""
    SELECT ville, id_mesure AS id_batiment, SUM(conso_kwh) AS conso_totale_kwh
    FROM conso
    GROUP BY ville, id_mesure
    ORDER BY ville, conso_totale_kwh DESC
""").createOrReplaceTempView("conso_par_batiment")

 Maintenant choisir top 2 par ville

In [ ]:
spark.sql("""
    SELECT ville, id_batiment, conso_totale_kwh
    FROM (
        SELECT ville, id_batiment, conso_totale_kwh,
               ROW_NUMBER() OVER (PARTITION BY ville ORDER BY conso_totale_kwh DESC) as rn
        FROM conso_par_batiment
    ) tmp
    WHERE rn <= 2
    ORDER BY ville, conso_totale_kwh DESC
""").show(truncate=False)

+----------+-----------+----------------+
|ville     |id_batiment|conso_totale_kwh|
+----------+-----------+----------------+
|Agadir    |25         |9404.3          |
|Agadir    |80         |9350.2          |
|Casablanca|143        |9466.5          |
|Casablanca|298        |9448.2          |
|Fès       |272        |9507.3          |
|Fès       |63         |9478.1          |
|Marrakech |121        |9969.1          |
|Marrakech |285        |9877.9          |
|Rabat     |84         |9849.9          |
|Rabat     |117        |9614.6          |
|Tanger    |70         |9328.0          |
|Tanger    |196        |8956.1          |
+----------+-----------+----------------+



Grâce à une requête SQL utilisant ROW_NUMBER(), le script identifie les deux bâtiments ayant la consommation la plus élevée dans chaque ville. Cette analyse met en évidence les bâtiments particuliers qui contribuent fortement à la consommation locale.

### **3) Comparaison DataFrame vs SQL**
 Exemple : consommation totale par type_batiment (DataFrame API)

In [ ]:
conso_par_type_df = df_time.groupBy("type_batiment").agg(_sum("conso_kwh").alias("conso_totale_kwh")) \
                          .orderBy(col("conso_totale_kwh").desc())
print("=== DataFrame API : consommation totale par type_batiment ===")
conso_par_type_df.show(truncate=False)

=== DataFrame API : consommation totale par type_batiment ===
+-------------+-----------------+
|type_batiment|conso_totale_kwh |
+-------------+-----------------+
|Résidentiel  |337156.8000000001|
|Bureau       |324889.1000000001|
|Hôpital      |287420.1000000001|
|Industriel   |277056.7         |
|École        |270936.4         |
+-------------+-----------------+



Le programme ajoute une colonne niveau_conso qui classe chaque mesure en trois catégories : faible, moyenne ou élevée. Cette classification rend l’analyse plus qualitative et facilite l’interprétation des comportements de consommation.

# **Exercice 6 : Agrégations conditionnelles**

### **1) Créer niveau_conso**

In [ ]:
df_levels = df_time.withColumn(
    "niveau_conso",
    when(col("conso_kwh") < 500, "Faible")
    .when((col("conso_kwh") >= 500) & (col("conso_kwh") < 2000), "Moyenne")
    .otherwise("Élevée")
)

Enfin, le script calcule la consommation moyenne, le coût total et le nombre de mesures pour chaque combinaison de niveau de consommation et de type de bâtiment. Cette analyse détaillée permet d’identifier les types de bâtiments qui génèrent les coûts les plus importants.

### **2) Calculer coût total et conso moyenne par niveau_conso et type_batiment**

In [ ]:
agg_levels = df_levels.groupBy("niveau_conso", "type_batiment").agg(
    _sum("cout_total").alias("cout_total_par_groupe"),
    _avg("conso_kwh").alias("conso_moy_kwh_par_groupe"),
    _count("*").alias("nb_mesures")
).orderBy("niveau_conso", col("cout_total_par_groupe").desc())

print("=== Agrégations par niveau_conso et type_batiment ===")
agg_levels.show(200, truncate=False)

=== Agrégations par niveau_conso et type_batiment ===
+------------+-------------+---------------------+------------------------+----------+
|niveau_conso|type_batiment|cout_total_par_groupe|conso_moy_kwh_par_groupe|nb_mesures|
+------------+-------------+---------------------+------------------------+----------+
|Faible      |Hôpital      |2033.6349999999998   |307.7                   |5         |
|Faible      |École        |1484.511             |329.175                 |4         |
|Faible      |Industriel   |1333.1490000000001   |383.26666666666665      |3         |
|Faible      |Résidentiel  |1255.7250000000001   |356.5                   |3         |
|Faible      |Bureau       |258.431              |228.7                   |1         |
|Moyenne     |Bureau       |24419.622            |1448.7692307692307      |13        |
|Moyenne     |Hôpital      |18554.281            |1470.39                 |10        |
|Moyenne     |Résidentiel  |17867.157            |1373.427272727273       |1